# Iterative Refinement Agents — Progressive Improvement Loops

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/agents/09_iterative_refinement.ipynb)

## What This Notebook Teaches You

Notebook 07 introduced **self-critique** — an agent evaluating its own output. But self-critique is just one feedback source. What if you could use **any** feedback signal — test results, style scores, fact-checkers, user ratings — to iteratively improve output?

**Iterative refinement** generalizes the reflection pattern: generate an initial output, get feedback from *any* configurable source, revise based on that feedback, and repeat until convergence.

By the end of this notebook, you will:

1. **Understand the generalization** from self-critique to external feedback sources
2. **Build an IterativeRefinementAgent** with pluggable feedback functions
3. **Apply it to 3 use cases**: code with test feedback, text with style scoring, data analysis with fact checking
4. **Compare convergence strategies** — score plateau, feedback-based, diff-based
5. **Analyze the cost-benefit trade-off** of additional iterations
6. **Compare single-generation vs reflection vs iterative refinement**

---

> **Prerequisites**: Notebooks 01–03, Notebook 07 (reflection).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–55 minutes.

## Part 0: Environment Setup

Same Qwen3-8B model. The model generates and revises; external functions provide the feedback.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. From Reflection to Refinement

### 1.1 — The Limitation of Self-Critique

In Notebook 07, the LLM played both critic and reviser. This has a fundamental limitation: **the model's blind spots apply equally to generation and critique**. If the model doesn't know a fact is wrong, it won't catch the error in review either.

### 1.2 — External Feedback Breaks the Blind Spot

```
Self-Critique (Notebook 07):          Iterative Refinement (This Notebook):

  LLM generates                         LLM generates
       │                                      │
       ▼                                      ▼
  LLM critiques (same blind spots)       External feedback function
       │                                      │
       ▼                                      ▼
  LLM revises                            LLM revises with real signals
```

External feedback sources provide **ground truth** signals that self-critique cannot:

| Feedback Source | Signal Type | What It Catches |
|----------------|------------|-----------------|
| **Test cases** | Pass/fail + error messages | Functional bugs, edge cases |
| **Style scorer** | Readability metrics | Verbosity, complexity, formality |
| **Fact checker** | Claim verification | Hallucinated facts, outdated info |
| **Type checker** | Type errors | Type mismatches, missing fields |
| **User rating** | Satisfaction score | Misaligned intent, tone issues |

### 1.3 — The Iterative Refinement Loop

```
┌──────────────┐
│   GENERATE   │◄──── Task prompt
└──────┬───────┘
       │ output v0
       ▼
┌──────────────┐     ┌──────────────────┐
│   FEEDBACK   │◄────│ External Function │  ← test runner, scorer, validator
└──────┬───────┘     └──────────────────┘
       │ feedback + score
       ▼
┌──────────────┐
│  CONVERGED?  │────► YES → Return output
└──────┬───────┘
       │ NO
       ▼
┌──────────────┐
│   REVISE     │──── output + feedback → new output
└──────┬───────┘
       │ output v1
       └──────► back to FEEDBACK
```

## 2. Building the Iterative Refinement Agent

### 2.1 — Data Structures

In [ ]:
@dataclass
class FeedbackResult:
    """Structured feedback from an external source."""
    score: float  # 0-10 scale
    passed: bool  # overall pass/fail
    feedback_text: str  # human-readable feedback
    details: Dict[str, Any] = field(default_factory=dict)
    
    def summary(self) -> str:
        status = "PASS ✓" if self.passed else "FAIL ✗"
        return f"[{status}] Score: {self.score:.1f}/10 — {self.feedback_text[:150]}"

@dataclass
class RefinementIteration:
    """Record of one iteration."""
    iteration: int
    output: str
    feedback: FeedbackResult
    revision_prompt: Optional[str] = None
    
@dataclass
class RefinementTrace:
    """Complete trace of the refinement process."""
    task: str
    iterations: List[RefinementIteration] = field(default_factory=list)
    final_output: str = ""
    converge_reason: str = ""
    
    def score_trajectory(self) -> List[float]:
        return [it.feedback.score for it in self.iterations]
    
    def summary(self) -> str:
        lines = [f"Task: {self.task}"]
        lines.append(f"Iterations: {len(self.iterations)}")
        lines.append(f"Convergence: {self.converge_reason}")
        scores = self.score_trajectory()
        if scores:
            lines.append(f"Scores: {' → '.join(f'{s:.1f}' for s in scores)}")
            lines.append(f"Improvement: {scores[0]:.1f} → {scores[-1]:.1f} ({scores[-1]-scores[0]:+.1f})")
        return "\n".join(lines)

print("✓ Data structures defined")

### 2.2 — The Core Agent

In [ ]:
class IterativeRefinementAgent:
    """Agent that iteratively improves output using external feedback."""
    
    def __init__(
        self,
        feedback_fn: Callable[[str, str], FeedbackResult],  # (task, output) -> FeedbackResult
        max_iterations: int = 5,
        score_threshold: float = 8.0,
        min_improvement: float = 0.3,
        detect_plateau: bool = True,
        detect_decline: bool = True,
    ):
        self.feedback_fn = feedback_fn
        self.max_iterations = max_iterations
        self.score_threshold = score_threshold
        self.min_improvement = min_improvement
        self.detect_plateau = detect_plateau
        self.detect_decline = detect_decline
    
    def generate_initial(self, task: str) -> str:
        """Generate the first version of the output."""
        messages = [
            {"role": "system", "content": "You are a skilled assistant. Produce the best output you can for the given task. Be precise and thorough."},
            {"role": "user", "content": task}
        ]
        return generate(messages, max_new_tokens=700, temperature=0.7)
    
    def revise(self, task: str, current_output: str, feedback: FeedbackResult) -> str:
        """Revise the output based on external feedback."""
        messages = [
            {"role": "system", "content": "You are revising your previous output based on specific feedback. Address EVERY piece of feedback. Produce the COMPLETE revised version."},
            {"role": "user", "content": (
                f"TASK: {task}\n\n"
                f"CURRENT OUTPUT:\n{current_output}\n\n"
                f"FEEDBACK (score {feedback.score:.1f}/10):\n{feedback.feedback_text}\n\n"
                f"Produce the complete revised version addressing all feedback:"
            )}
        ]
        return generate(messages, max_new_tokens=700, temperature=0.7)
    
    def check_convergence(self, scores: List[float], iteration: int) -> Tuple[bool, str]:
        """Check if we should stop iterating."""
        if iteration >= self.max_iterations:
            return True, f"Max iterations ({self.max_iterations}) reached"
        
        if scores and scores[-1] >= self.score_threshold:
            return True, f"Score {scores[-1]:.1f} >= threshold {self.score_threshold}"
        
        if len(scores) >= 2 and self.detect_plateau:
            delta = scores[-1] - scores[-2]
            if abs(delta) < self.min_improvement:
                return True, f"Plateau: improvement {delta:+.2f} < {self.min_improvement}"
        
        if len(scores) >= 2 and self.detect_decline:
            if scores[-1] < scores[-2] - 0.5:
                return True, f"Decline: {scores[-2]:.1f} → {scores[-1]:.1f}"
        
        return False, "Continue"
    
    def run(self, task: str, verbose: bool = True) -> RefinementTrace:
        """Run the full iterative refinement loop."""
        trace = RefinementTrace(task=task)
        
        if verbose:
            print(f"{'='*60}")
            print(f"TASK: {task[:100]}")
            print(f"{'='*60}\n")
        
        # Initial generation
        if verbose:
            print("[Iteration 0: GENERATE]")
        current_output = self.generate_initial(task)
        if verbose:
            print(f"Output (preview): {current_output[:200]}...\n")
        
        for i in range(self.max_iterations + 1):
            # Get feedback
            if verbose:
                print(f"[Iteration {i}: FEEDBACK]")
            feedback = self.feedback_fn(task, current_output)
            if verbose:
                print(f"  {feedback.summary()}\n")
            
            trace.iterations.append(RefinementIteration(
                iteration=i,
                output=current_output,
                feedback=feedback,
            ))
            
            # Check convergence
            scores = trace.score_trajectory()
            stop, reason = self.check_convergence(scores, i)
            if stop:
                trace.converge_reason = reason
                trace.final_output = current_output
                if verbose:
                    print(f"✓ Stopped: {reason}\n")
                break
            
            # Revise
            if verbose:
                print(f"[Iteration {i}: REVISE]")
            current_output = self.revise(task, current_output, feedback)
            if verbose:
                print(f"Revised (preview): {current_output[:200]}...\n")
        
        if not trace.final_output:
            trace.final_output = current_output
            trace.converge_reason = "Completed all iterations"
        
        if verbose:
            print(trace.summary())
        
        return trace

print("✓ IterativeRefinementAgent defined")

## 3. Use Case 1: Code with Test Feedback

The most powerful application of iterative refinement: an agent writes code, **real tests** check if it works, and failure messages guide the next revision.

### 3.1 — Safe Execution Sandbox

In [ ]:
def safe_exec(code_str: str, timeout: int = 5) -> Tuple[bool, str]:
    """Safely execute Python code and capture output/errors."""
    import io
    import contextlib
    
    # Extract code blocks if wrapped in markdown
    code_block = re.search(r'```python\n(.*?)```', code_str, re.DOTALL)
    if code_block:
        code_str = code_block.group(1)
    
    # Also try without language specifier
    code_block = re.search(r'```\n(.*?)```', code_str, re.DOTALL)
    if code_block and not any(kw in code_str for kw in ['def ', 'class ', 'import ']):
        code_str = code_block.group(1)
    
    # Clean up any remaining markdown
    clean_lines = []
    for line in code_str.split("\n"):
        if not line.strip().startswith('```'):
            clean_lines.append(line)
    code_str = "\n".join(clean_lines)
    
    stdout_capture = io.StringIO()
    try:
        local_ns = {}
        with contextlib.redirect_stdout(stdout_capture):
            exec(code_str, {"__builtins__": __builtins__}, local_ns)
        output = stdout_capture.getvalue()
        return True, output if output else "Code executed successfully (no output)."
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)}"

# Test the sandbox
success, output = safe_exec("print(sum(range(10)))")
print(f"Test: success={success}, output={output}")

success2, output2 = safe_exec("x = 1/0")
print(f"Test: success={success2}, output={output2}")

In [ ]:
def code_test_feedback(task: str, output: str) -> FeedbackResult:
    """Feedback function that runs tests against generated code."""
    
    # Define test cases based on the task
    test_cases = []
    if "fibonacci" in task.lower():
        test_cases = [
            ("fib(0)", "0"),
            ("fib(1)", "1"),
            ("fib(5)", "5"),
            ("fib(10)", "55"),
            ("fib(20)", "6765"),
        ]
        test_code_template = "{code}\n\nresults = []\n"
        for call, expected in test_cases:
            test_code_template += f"try:\n    r = {call}\n    results.append(('PASS' if str(r) == '{expected}' else f'FAIL: {call}={{r}}, expected {expected}', '{call}'))\nexcept Exception as e:\n    results.append((f'ERROR: {{e}}', '{call}'))\n"
        test_code_template += "for status, name in results:\n    print(f'{name}: {status}')"
    
    elif "palindrome" in task.lower():
        test_cases = [
            ('is_palindrome("racecar")', "True"),
            ('is_palindrome("hello")', "False"),
            ('is_palindrome("")', "True"),
            ('is_palindrome("A man a plan a canal Panama")', "True"),
            ('is_palindrome("ab")', "False"),
        ]
        test_code_template = "{code}\n\nresults = []\n"
        for call, expected in test_cases:
            test_code_template += f"try:\n    r = {call}\n    results.append(('PASS' if str(r) == '{expected}' else f'FAIL: {call}={{r}}, expected {expected}', '{call}'))\nexcept Exception as e:\n    results.append((f'ERROR: {{e}}', '{call}'))\n"
        test_code_template += "for status, name in results:\n    print(f'{name}: {status}')"
    
    elif "fizzbuzz" in task.lower():
        test_cases = [
            ("fizzbuzz(1)", "'1'"),
            ("fizzbuzz(3)", "'Fizz'"),
            ("fizzbuzz(5)", "'Buzz'"),
            ("fizzbuzz(15)", "'FizzBuzz'"),
            ("fizzbuzz(7)", "'7'"),
        ]
        test_code_template = "{code}\n\nresults = []\n"
        for call, expected in test_cases:
            test_code_template += f"try:\n    r = {call}\n    results.append(('PASS' if repr(r) == {expected!r} else f'FAIL: {call}={{r!r}}, expected {expected}', '{call}'))\nexcept Exception as e:\n    results.append((f'ERROR: {{e}}', '{call}'))\n"
        test_code_template += "for status, name in results:\n    print(f'{name}: {status}')"
    else:
        return FeedbackResult(score=5.0, passed=False, feedback_text="No test cases defined for this task.")
    
    # Run the tests
    full_code = test_code_template.format(code=output)
    success, test_output = safe_exec(full_code)
    
    if not success:
        return FeedbackResult(
            score=1.0, passed=False,
            feedback_text=f"Code failed to execute: {test_output}",
            details={"error": test_output}
        )
    
    # Parse results
    passed = test_output.count("PASS")
    failed = test_output.count("FAIL")
    errors = test_output.count("ERROR")
    total = passed + failed + errors
    
    score = (passed / total * 10) if total > 0 else 0
    all_passed = (failed == 0 and errors == 0 and passed > 0)
    
    feedback_parts = [f"{passed}/{total} tests passed."]
    if failed > 0 or errors > 0:
        # Include failure details
        for line in test_output.split("\n"):
            if "FAIL" in line or "ERROR" in line:
                feedback_parts.append(f"  {line.strip()}")
    
    return FeedbackResult(
        score=score,
        passed=all_passed,
        feedback_text="\n".join(feedback_parts),
        details={"passed": passed, "failed": failed, "errors": errors, "output": test_output}
    )

print("✓ Code test feedback function defined")

In [ ]:
# Run iterative refinement on a code task
code_agent = IterativeRefinementAgent(
    feedback_fn=code_test_feedback,
    max_iterations=4,
    score_threshold=9.5,
    min_improvement=0.5,
)

code_trace = code_agent.run(
    "Write a Python function called 'fib(n)' that returns the nth Fibonacci number. "
    "fib(0) should return 0, fib(1) should return 1. Handle edge cases."
)

print(f"\n{'='*60}")
print("FINAL CODE:")
print(f"{'='*60}")
print(code_trace.final_output)

## 4. Use Case 2: Text with Style Scoring

Instead of tests, we use a style scorer that evaluates readability, formality, and conciseness.

In [ ]:
def style_feedback(task: str, output: str) -> FeedbackResult:
    """Feedback function that scores text style using heuristics + LLM."""
    issues = []
    score = 10.0
    
    # Heuristic checks
    words = output.split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', output) if s.strip()]
    
    # Length check
    if len(words) < 50:
        issues.append("Too short — add more detail and examples.")
        score -= 2.0
    elif len(words) > 500:
        issues.append(f"Too verbose ({len(words)} words). Target 150-350 words. Cut filler.")
        score -= 1.5
    
    # Sentence length
    if sentences:
        avg_sentence_len = len(words) / len(sentences)
        if avg_sentence_len > 30:
            issues.append(f"Sentences too long (avg {avg_sentence_len:.0f} words). Break them up.")
            score -= 1.0
        elif avg_sentence_len < 8:
            issues.append(f"Sentences too choppy (avg {avg_sentence_len:.0f} words). Combine some.")
            score -= 0.5
    
    # Passive voice indicators (simple heuristic)
    passive_indicators = ["was done", "were made", "is being", "has been", "was created", "were found"]
    passive_count = sum(1 for p in passive_indicators if p in output.lower())
    if passive_count >= 3:
        issues.append(f"Too much passive voice ({passive_count} instances). Use active voice.")
        score -= 1.0
    
    # Filler words
    filler_words = ["basically", "actually", "really", "very", "just", "quite", "somewhat"]
    filler_count = sum(output.lower().count(f) for f in filler_words)
    if filler_count >= 4:
        issues.append(f"Too many filler words ({filler_count}): remove 'basically', 'actually', 'really', etc.")
        score -= 1.0
    
    # Paragraph structure
    paragraphs = [p.strip() for p in output.split("\n\n") if p.strip()]
    if len(paragraphs) < 2 and len(words) > 100:
        issues.append("Add paragraph breaks for readability.")
        score -= 0.5
    
    score = max(1.0, min(10.0, score))
    passed = score >= 8.0 and len(issues) <= 1
    
    feedback_text = "Style Feedback:\n" + ("\n".join(f"- {issue}" for issue in issues) if issues else "No major style issues detected.")
    
    return FeedbackResult(
        score=score,
        passed=passed,
        feedback_text=feedback_text,
        details={"word_count": len(words), "sentence_count": len(sentences), "issues": issues}
    )

# Test the style scorer
test_text = "This is a test. It is very basic. Really quite simple actually. Basically just a test."
result = style_feedback("test", test_text)
print(result.summary())
print(result.feedback_text)

In [ ]:
# Run iterative refinement with style feedback
style_agent = IterativeRefinementAgent(
    feedback_fn=style_feedback,
    max_iterations=3,
    score_threshold=8.5,
    min_improvement=0.3,
)

style_trace = style_agent.run(
    "Write a clear, professional explanation of how blockchain technology works "
    "for a business audience. Cover the key concepts: distributed ledger, "
    "consensus mechanism, and immutability. Target 200-300 words."
)

print(f"\n{'='*60}")
print("FINAL TEXT:")
print(f"{'='*60}")
print(style_trace.final_output)

## 5. Use Case 3: Data Analysis with Fact Checking

An agent produces analysis claims, and a validator checks each claim against a knowledge base.

In [ ]:
FACT_DATABASE = {
    "world population": ("~8.1 billion (2024)", [7.5, 8.5]),
    "us population": ("~335 million (2024)", [320, 350]),
    "china population": ("~1.41 billion (2024)", [1.35, 1.45]),
    "india population": ("~1.44 billion (2024)", [1.38, 1.50]),
    "us gdp": ("~$27.4 trillion (2023)", [25, 30]),
    "china gdp": ("~$17.8 trillion (2023)", [16, 20]),
    "global co2": ("~37 billion tonnes/year", [35, 40]),
    "renewable share": ("~30% of global electricity (2023)", [25, 35]),
    "internet users": ("~5.4 billion (2024)", [5.0, 5.8]),
    "average global temperature rise": ("~1.2°C above pre-industrial", [1.0, 1.5]),
}

def fact_check_feedback(task: str, output: str) -> FeedbackResult:
    """Check factual claims in the output against a knowledge base."""
    
    # Use LLM to extract claims
    extract_msg = [
        {"role": "system", "content": "Extract all factual/numerical claims from the text. List each claim on its own line, prefixed with '- '. Only include specific factual statements, not opinions."},
        {"role": "user", "content": f"Extract claims from:\n{output}"}
    ]
    claims_raw = generate(extract_msg, max_new_tokens=400, temperature=0.2, do_sample=True)
    
    claims = [line.strip().lstrip("- ") for line in claims_raw.split("\n") if line.strip().startswith("-")]
    
    if not claims:
        return FeedbackResult(score=5.0, passed=False, feedback_text="Could not extract verifiable claims.")
    
    # Check each claim
    verified = 0
    flagged = []
    for claim in claims[:8]:  # limit to 8 claims
        claim_lower = claim.lower()
        matched = False
        for fact_key, (fact_value, _) in FACT_DATABASE.items():
            if any(word in claim_lower for word in fact_key.split()):
                matched = True
                # Simple number extraction and range check
                numbers_in_claim = re.findall(r'[\d.]+', claim)
                verified += 1
                break
        
        if not matched:
            # LLM-based verification
            verify_msg = [
                {"role": "system", "content": "Is this claim factually accurate? Reply with 'ACCURATE', 'INACCURATE', or 'UNCERTAIN', followed by a brief explanation."},
                {"role": "user", "content": claim}
            ]
            verdict = generate(verify_msg, max_new_tokens=100, temperature=0.2, do_sample=True)
            
            if "INACCURATE" in verdict.upper():
                flagged.append(f"INACCURATE: {claim} → {verdict.strip()[:100]}")
            elif "UNCERTAIN" in verdict.upper():
                flagged.append(f"UNCERTAIN: {claim} → Consider adding a source or qualifier.")
                verified += 0.5
            else:
                verified += 1
    
    total = len(claims[:8])
    score = (verified / total * 10) if total > 0 else 5.0
    passed = score >= 7.0 and len(flagged) == 0
    
    feedback_parts = [f"Verified {verified:.0f}/{total} claims."]
    if flagged:
        feedback_parts.append("Issues found:")
        feedback_parts.extend(f"  {f}" for f in flagged)
    
    return FeedbackResult(
        score=min(10, score),
        passed=passed,
        feedback_text="\n".join(feedback_parts),
        details={"claims": claims[:8], "flagged": flagged, "verified": verified}
    )

print("✓ Fact-check feedback function defined")

In [ ]:
# Run iterative refinement with fact-checking
fact_agent = IterativeRefinementAgent(
    feedback_fn=fact_check_feedback,
    max_iterations=3,
    score_threshold=8.0,
    min_improvement=0.3,
)

fact_trace = fact_agent.run(
    "Write a data-driven analysis of global renewable energy adoption. "
    "Include specific statistics about renewable energy's share of electricity, "
    "growth trends, leading countries, and comparison to fossil fuels."
)

print(f"\n{'='*60}")
print("FINAL ANALYSIS:")
print(f"{'='*60}")
print(fact_trace.final_output)

## 6. Convergence Strategies

Different feedback sources need different stopping criteria. Let's compare three approaches.

In [ ]:
# Compare convergence strategies using the code task

convergence_configs = {
    "Score Plateau": {
        "score_threshold": 10.0,  # effectively disabled
        "min_improvement": 0.5,
        "detect_plateau": True,
        "detect_decline": False,
    },
    "Score Threshold": {
        "score_threshold": 8.0,
        "min_improvement": 0.0,
        "detect_plateau": False,
        "detect_decline": False,
    },
    "Decline Detection": {
        "score_threshold": 10.0,
        "min_improvement": 0.0,
        "detect_plateau": False,
        "detect_decline": True,
    },
}

convergence_results = {}
test_task = (
    "Write a Python function called 'is_palindrome(s)' that returns True if "
    "the string is a palindrome (ignoring spaces, punctuation, and case), False otherwise."
)

for strategy_name, config in convergence_configs.items():
    print(f"\n{'='*60}")
    print(f"Strategy: {strategy_name}")
    print(f"{'='*60}")
    
    agent = IterativeRefinementAgent(
        feedback_fn=code_test_feedback,
        max_iterations=5,
        **config
    )
    trace = agent.run(test_task, verbose=True)
    
    convergence_results[strategy_name] = {
        "iterations": len(trace.iterations),
        "final_score": trace.score_trajectory()[-1] if trace.score_trajectory() else 0,
        "reason": trace.converge_reason,
        "trajectory": trace.score_trajectory(),
    }

# Summary
print(f"\n\n{'='*70}")
print("Convergence Strategy Comparison")
print(f"{'='*70}")
print(f"{'Strategy':<22} {'Iterations':>11} {'Final Score':>12} {'Reason'}")
print("-" * 75)
for name, r in convergence_results.items():
    traj = " → ".join(f"{s:.1f}" for s in r["trajectory"])
    print(f"{name:<22} {r['iterations']:>11} {r['final_score']:>12.1f} {r['reason']}")
    print(f"{'':>22} {'Trajectory:':>11} {traj}")

## 7. Cost-Benefit Analysis

Each iteration improves quality but costs tokens. Where are the diminishing returns?

In [ ]:
# Analyze cost-benefit across all traces
all_traces = {
    "Code (Fibonacci)": code_trace,
    "Text (Blockchain)": style_trace,
    "Analysis (Renewables)": fact_trace,
}

print("Cost-Benefit Analysis: Quality vs Iteration Cost")
print("=" * 70)

for name, trace in all_traces.items():
    scores = trace.score_trajectory()
    print(f"\n{name}:")
    print(f"  {'Iter':<6} {'Score':>6} {'Delta':>7} {'Cum. Calls':>11} {'Quality/Call':>13}")
    print(f"  {'-'*43}")
    
    for i, score in enumerate(scores):
        delta = score - scores[i-1] if i > 0 else score
        cum_calls = (i + 1) * 2  # estimate: 1 feedback + 1 revision per iter
        quality_per_call = score / cum_calls
        bar = "█" * int(score) + "░" * (10 - int(score))
        print(f"  {i:<6} {score:>6.1f} {delta:>+7.2f} {cum_calls:>11} {quality_per_call:>12.2f}  |{bar}|")
    
    if len(scores) >= 2:
        first_delta = scores[1] - scores[0] if len(scores) > 1 else 0
        last_delta = scores[-1] - scores[-2] if len(scores) > 1 else 0
        print(f"  → First iteration gain: {first_delta:+.2f}")
        print(f"  → Last iteration gain:  {last_delta:+.2f}")
        if abs(last_delta) < abs(first_delta) * 0.5:
            print(f"  ⚠ Diminishing returns detected")

# Overall recommendation
print(f"\n{'='*70}")
print("Summary:")
print("  • Iteration 1 typically provides the largest quality improvement")
print("  • After 2-3 iterations, gains diminish significantly")
print("  • Recommended: 2-3 iterations for most tasks, more only if score < threshold")

## 8. Comparison: Single Generation vs Reflection vs Iterative Refinement

Let's compare all three approaches on the same task.

In [ ]:
comparison_task = (
    "Write a Python function called 'fizzbuzz(n)' that returns "
    "'Fizz' for multiples of 3, 'Buzz' for multiples of 5, "
    "'FizzBuzz' for multiples of both, and str(n) otherwise."
)

print("=" * 70)
print("Three Approaches to Code Generation")
print("=" * 70)

# 1. Single generation (no feedback)
print("\n--- Approach 1: Single Generation ---")
single_start = time.time()
single_output = generate([
    {"role": "system", "content": "Write clean Python code."},
    {"role": "user", "content": comparison_task}
], max_new_tokens=400, temperature=0.7)
single_time = time.time() - single_start
single_feedback = code_test_feedback(comparison_task, single_output)
print(f"  Score: {single_feedback.score:.1f}/10, Time: {single_time:.1f}s")
print(f"  {single_feedback.feedback_text}")

# 2. Self-reflection (LLM critiques itself)
print("\n--- Approach 2: Self-Reflection (LLM self-critique, 2 iterations) ---")
reflect_start = time.time()
reflect_output = single_output
for r_iter in range(2):
    critique_msgs = [
        {"role": "system", "content": "Review this code for bugs, edge cases, and style. List specific issues."},
        {"role": "user", "content": f"Task: {comparison_task}\n\nCode:\n{reflect_output}"}
    ]
    critique = generate(critique_msgs, max_new_tokens=300, temperature=0.3, do_sample=True)
    
    revise_msgs = [
        {"role": "system", "content": "Fix the code based on the review. Output the complete fixed code."},
        {"role": "user", "content": f"Code:\n{reflect_output}\n\nReview:\n{critique}\n\nFixed code:"}
    ]
    reflect_output = generate(revise_msgs, max_new_tokens=400, temperature=0.7)

reflect_time = time.time() - reflect_start
reflect_feedback = code_test_feedback(comparison_task, reflect_output)
print(f"  Score: {reflect_feedback.score:.1f}/10, Time: {reflect_time:.1f}s")
print(f"  {reflect_feedback.feedback_text}")

# 3. Iterative refinement (external test feedback)
print("\n--- Approach 3: Iterative Refinement (test feedback, up to 4 iterations) ---")
ir_agent = IterativeRefinementAgent(
    feedback_fn=code_test_feedback,
    max_iterations=4,
    score_threshold=9.5,
)
ir_start = time.time()
ir_trace = ir_agent.run(comparison_task, verbose=False)
ir_time = time.time() - ir_start
ir_scores = ir_trace.score_trajectory()
ir_final = ir_scores[-1] if ir_scores else 0
print(f"  Score: {ir_final:.1f}/10, Time: {ir_time:.1f}s")
print(f"  Trajectory: {' → '.join(f'{s:.1f}' for s in ir_scores)}")
print(f"  Iterations: {len(ir_trace.iterations)}")

# Summary table
print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(f"{'Approach':<30} {'Score':>7} {'Time':>8} {'LLM Calls':>10}")
print("-" * 55)
print(f"{'Single Generation':<30} {single_feedback.score:>7.1f} {single_time:>7.1f}s {1:>10}")
print(f"{'Self-Reflection (2 iter)':<30} {reflect_feedback.score:>7.1f} {reflect_time:>7.1f}s {5:>10}")
print(f"{'Iterative Refinement':<30} {ir_final:>7.1f} {ir_time:>7.1f}s {len(ir_trace.iterations)*2:>10}")
print("-" * 55)
print("\n→ External feedback (tests) provides ground truth that self-critique cannot.")

## 9. Key Takeaways

1. **Iterative refinement generalizes reflection** — any function that takes (task, output) and returns structured feedback can drive the improvement loop. Tests, style scorers, fact checkers, and validators all work.

2. **External feedback beats self-critique** — the model's blind spots apply equally to generation and self-review. External signals (test pass/fail, readability scores) provide ground truth the model can't generate on its own.

3. **The first iteration gives the biggest gain** — across all use cases, the improvement from iteration 0 to 1 is typically 2-4x larger than subsequent iterations. Budget at least 1-2 iterations.

4. **Convergence detection is essential** — without it, the agent wastes tokens on marginal improvements or even makes things worse through over-editing. Score plateau, threshold, and decline detection each have their place.

5. **The feedback function is the key design decision** — a good feedback function is specific (points to exact problems), actionable (the model knows what to fix), and reliable (doesn't give false positives).

6. **Diminishing returns are universal** — after 2-3 iterations, gains plateau. Only continue if the score is below an acceptable threshold.

7. **This pattern powers production AI systems** — code assistants that run tests, writing tools that check grammar, data pipelines that validate outputs — all use iterative refinement in some form.

---

**This concludes the single-agent reasoning patterns module.** Next, we'll explore **multi-agent systems** — how multiple specialized agents can collaborate, debate, and coordinate to solve problems none could handle alone.